# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` — folds, track preparation, metrics, the loop itself
- `src/models/` — model functions and `MODEL_REGISTRY`
- `src/features/quantum/` — quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "david-eval"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: /home/claude/manuelarceaguirre/capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = "David"  # your name, recorded on every result row (default: OS user)
RESUME        = False # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce"]
ACTIVE_TRACKS = ["B"]                                          # any keys of TRACK_CONFIG
ACTIVE_MODELS = ["linear_svm", "sig_kernel"]                   # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['B']
Models:   ['linear_svm', 'sig_kernel']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track B        target INDPRO     loaded   shape=(767, 10)
  Track B        target PAYEMS     loaded   shape=(767, 10)
  Track B        target CPIAUCSL   loaded   shape=(767, 12)
  Track B        target S&P 500    loaded   shape=(767, 12)
Stationary panel (767, 123)  1962-04-01 -> 2026-02-01
Targets          (806, 20)


In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Saved: raw/linear_svm/B.parquet   new=300  total=300
  Saved: raw/sig_kernel/B.parquet   new=300  total=300


  1000/15528 runs complete · 2m23s elapsed · ~34m41s left


  Saved: raw/linear_svm/B.parquet   new=300  total=600
  Saved: raw/sig_kernel/B.parquet   new=300  total=600


  Saved: raw/linear_svm/B.parquet   new=300  total=900
  Saved: raw/sig_kernel/B.parquet   new=300  total=900


  2000/15528 runs complete · 4m45s elapsed · ~32m05s left


...

  Saved: raw/linear_svm/B.parquet   new=300  total=6900
  Saved: raw/sig_kernel/B.parquet   new=300  total=6900


  14000/15528 runs complete · 33m18s elapsed · ~3m38s left


  Saved: raw/linear_svm/B.parquet   new=300  total=7200
  Saved: raw/sig_kernel/B.parquet   new=300  total=7200


  15000/15528 runs complete · 35m36s elapsed · ~1m15s left
  Saved: raw/linear_svm/B.parquet   new=300  total=7500
  Saved: raw/sig_kernel/B.parquet   new=300  total=7500


  Saved: raw/linear_svm/B.parquet   new=224  total=7724
  Saved: raw/sig_kernel/B.parquet   new=224  total=7724
Loop complete. Total rows: 15448 in 36m48s
Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae      rmse  \
model      track target   horizon                                            
linear_svm B     CPIAUCSL 1        0.000010  0.002206   0.002206  0.002206   
                          3        0.000067  0.005564   0.005564  0.005564   
                          6        0.000230  0.010247   0.010247  0.010247   
                 INDPRO   1        0.000085  0.005524   0.005524  0.005524   
                          3        0.000383  0.011889   0.011889  0.011889   
                          6        0.001116  0.021909   0.021909  0.021909   
                 PAYEMS   1        0.000069  0.001626   0.001626  0.001626   
                          3        0.000104  0.003416   0.003416  0.003416   
                          6        0.000234  0.007372   0.007372  0.007372   
                 S&P 500  1        0.001700  0.027679   0.027679  0.027679   
                          3        0.009369  0.058503   0.058503  0.058503   
                          6        0.025211  0.094178   0.094178  0.094178   
sig_kernel B     CPIAUCSL 1        0.000006  0.001757   0.001757  0.001757   
                          3        0.000042  0.004524   0.004524  0.004524   
                          6        0.000136  0.008206   0.008206  0.008206   
                 INDPRO   1        0.000097  0.005430   0.005430  0.005430   
                          3        0.000430  0.011591   0.011591  0.011591   
                          6        0.000967  0.019495   0.019495  0.019495   
                 PAYEMS   1        0.000045  0.001432   0.001432  0.001432   
                          3        0.000161  0.004159   0.004159  0.004159   
                          6        0.000272  0.007594   0.007594  0.007594   
                 S&P 500  1        0.001493  0.028061   0.028061  0.028061   
                          3        0.006011  0.056461   0.056461  0.056461   
                          6        0.013844  0.084419   0.084419  0.084419   

                                       mase      rmspe  
model      track target   horizon                       
linear_svm B     CPIAUCSL 1        1.165740   1.366023  
                          3        1.195405   2.799949  
                          6        1.452867   1.438645  
                 INDPRO   1        0.871339   8.139526  
                          3        0.997762   7.027343  
                          6        1.064469  66.405126  
                 PAYEMS   1        1.211266   1.682398  
                          3        1.272800   1.240977  
                          6        1.368394   2.075698  
                 S&P 500  1        0.841481   4.524817  
                          3        0.845778   2.109494  
                          6        0.961370   2.612467  
sig_kernel B     CPIAUCSL 1        0.929354   1.672393  
                          3        0.959991   2.759205  
                          6        1.148650   1.250424  
                 INDPRO   1        0.857034   4.809355  
                          3        0.964445   5.603928  
                          6        0.956775  16.231089  
                 PAYEMS   1        1.098264   1.434009  
                          3        1.355484   1.466169  
                          6        1.359192   1.953670  
                 S&P 500  1        0.854163   5.031203  
                          3        0.815190   2.233702  
                          6        0.836501   2.123339

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  median_ae  \
model      track target   horizon usrec                                  
AR6        B     CPIAUCSL 1       0      0.000005  0.001586   0.001586   
                                  1      0.000019  0.003242   0.003242   
                          3       0      0.000027  0.003785   0.003785   
                                  1      0.000161  0.009629   0.009629   
                          6       0      0.000093  0.006961   0.006961   
...                                           ...       ...        ...   
sig_kernel B     S&P 500  1       1      0.003721  0.047260   0.047260   
                          3       0      0.004644  0.050757   0.050757   
                                  1      0.016535  0.100401   0.100401   
                          6       0      0.008608  0.071010   0.071010   
                                  1      0.053961  0.187161   0.187161   

                                             rmse      mase     rmspe  
model      track target   horizon usrec                                
AR6        B     CPIAUCSL 1       0      0.001586  0.860288  1.348996  
                                  1      0.003242  1.546022  1.333686  
                          3       0      0.003785  0.823602  2.077020  
                                  1      0.009629  1.925051  1.826281  
                          6       0      0.006961  1.001412  1.211426  
...                                           ...       ...       ...  
sig_kernel B     S&P 500  1       1      0.047260  1.354654  1.924388  
                          3       0      0.050757  0.738611  2.359688  
                                  1      0.100401  1.405050  1.263275  
                          6       0      0.071010  0.716677  2.236696  
                                  1      0.187161  1.754611  1.257836  

[504 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,linear_svm,B,CPIAUCSL,1,0,1972-04-01,0.002407,0.003103
1,linear_svm,B,CPIAUCSL,1,1,1972-05-01,0.002401,0.002868
2,linear_svm,B,CPIAUCSL,1,2,1972-06-01,0.002395,0.002575
3,linear_svm,B,CPIAUCSL,1,3,1972-07-01,0.002389,0.002729
4,linear_svm,B,CPIAUCSL,1,4,1972-08-01,0.004762,0.002249


,,mean,sum
model,track,,
linear_svm,B,0.016,125.701
sig_kernel,B,0.262,2021.846
